## Итоговые результаты

| Вариант | OOF AUC | Public LB |
|---|---|---|
| одиночный CatBoost (бейзлайн) | 0.834 | — |
| ансамбль 3 моделей, ранговый бленд | 0.8366 | 0.7688 |
| **ансамбль + seed-bagging** | 0.8374 | 0.7698 |

На деле было сильно больше вариантов, оставил только наиболее релевантные.

Тест продолжает train по времени. После ряда итераций отказались от target encoding
и тяжёлых произведений признаков (поднимали OOF, но не лидерборд), перешли на
ранговый бленд трёх разных бустингов и добавили seed-bagging.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

SEED, N_FOLDS = 42, 5
ID, TARGET = "front_id", "target_value"
CAT = ["db_group_last", "fl_adminarea"]
DATA_DIR = "."
BAG_SEEDS = [42, 202, 707]
USE_TE = False  # target encoding пробовал — на тесте просел, выключил

## 1. Загрузка данных

In [2]:
train = pd.read_csv(f"{DATA_DIR}/train_apps.csv")
test = pd.read_csv(f"{DATA_DIR}/test_apps.csv")
sample = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

train = train[train[TARGET].notna()].reset_index(drop=True)
train[TARGET] = train[TARGET].astype(int)
print("train", train.shape, "| test", test.shape, "| sample", sample.shape)
print("доля согласий:", round(train[TARGET].mean(), 4))

train (145241, 28) | test (36311, 27) | sample (45032, 2)
доля согласий: 0.0609


## 2. Обзор: сдвиг во времени

Тест лежит позже train, доля согласий растёт от периода к периоду. Часть id из
сабмита есть только в train — для них позже подставим out-of-fold прогнозы.

In [3]:
print("train decision_day:", train.decision_day.min(), "->", train.decision_day.max())
print("test  decision_day:", test.decision_day.min(), "->", test.decision_day.max())

d = pd.to_datetime(train["decision_day"])
per = pd.qcut((d - d.min()).dt.days, 5, labels=False)
print("\nдоля согласий по периодам:")
print(train.assign(p=per).groupby("p")[TARGET].mean().round(4))

need_overlap = set(sample[ID]) - set(test[ID])
print("\nid в сабмите:", len(sample), "| из test:", test[ID].nunique(), "| только в train:", len(need_overlap))

train decision_day: 2024-02-01 -> 2025-06-05
test  decision_day: 2025-06-05 -> 2025-12-01

доля согласий по периодам:
p
0    0.0309
1    0.0415
2    0.0548
3    0.0726
4    0.1052
Name: target_value, dtype: float64

id в сабмите: 45032 | из test: 36311 | только в train: 8721


## 3. Признаки

Облегчённый набор: даты, наценка над ставкой ЦБ, разности лимитов и активности,
счётчик пропусков. Тяжёлые произведения признаков убрал, потому что переобучали под прошлый
период.

In [4]:
def make_features(df):
    df = df.copy()
    d = pd.to_datetime(df["decision_day"])
    df["month"], df["dow"], df["day"] = d.dt.month, d.dt.dayofweek, d.dt.day
    df["date_ord"] = (d - pd.Timestamp("2024-02-01")).dt.days  # порядковый день — тренд
    df["rate_spread"] = df["offered_rate"] - df["cb_rate"]      # наценка банка
    df["od_limit_range"] = df["overdraft_limit_max"] - df["overdraft_limit_min"]
    df["od_max_vs_loan"] = df["overdraft_limit_max"] - df["loan_amount_last"]
    df["od_min_vs_loan"] = df["overdraft_limit_min"] - df["loan_amount_last"]
    df["deb_30_90_ratio"] = df["cnt_deb_ul_ip_30"] - df["cnt_deb_ul_ip_90"]
    df["sum_deb_30_90"] = df["sum_deb_ul_30"] - df["sum_deb_ul_90"]
    base_num = ["corp_credit_products", "sum_deb_ul_90", "sum_deb_ul_30", "cnt_deb_loan_90",
                "cnt_deb_ul_ip_90", "cnt_deb_ul_ip_30", "balance_rur_amt_30_min", "cnt_cred_loan_90",
                "days_from_authperson_registration", "fl_hdb_bki_total_active_products",
                "count_all_corp_dashboard_events", "p75_time_spent_minutes", "sum_deb_investment_90"]
    df["n_missing"] = df[base_num].isna().sum(axis=1)
    return df

train = make_features(train)
test = make_features(test)
feats_v1 = [c for c in train.columns if c not in [ID, TARGET, "decision_day"]]

In [5]:
# target encoding — оставлен как опция; в финал не вошёл (USE_TE=False)
def target_encode_cv(col, smoothing=50):
    gmean = train[TARGET].mean()
    enc = np.full(len(train), gmean)
    for tri, vai in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(train, train[TARGET]):
        s = train.iloc[tri].groupby(col)[TARGET].agg(["mean", "count"])
        m = (s["mean"] * s["count"] + gmean * smoothing) / (s["count"] + smoothing)
        enc[vai] = train.iloc[vai][col].map(m).fillna(gmean).values
    return enc

te_feats = []
if USE_TE:
    for col in ["fl_adminarea", "db_group_last", "month"]:
        train[f"{col}_te"] = target_encode_cv(col)
        te_feats.append(f"{col}_te")
feats_v3 = feats_v1 + te_feats
feats = feats_v3 if USE_TE else feats_v1
print("признаков:", len(feats), "| target encoding:", USE_TE)

признаков: 36 | target encoding: False


In [6]:
# категории отдаём моделям нативно (раньше кодировали целыми числами)
ref = {c: pd.Categorical(pd.concat([train[c], test[c]])).categories for c in CAT}
for c in CAT:
    train[c] = pd.Categorical(train[c], categories=ref[c])
    test[c] = pd.Categorical(test[c], categories=ref[c])

y = train[TARGET]
spw = (y == 0).sum() / (y == 1).sum()  # дисбаланс классов
X1tr, X1te = train[feats], test[feats]

## 4. Проверка на временном холдауте

Учим на ранних данных, валидируем на поздних — проверка, что модель держит сдвиг.

In [7]:
lgb_params = dict(objective="binary", metric="auc", learning_rate=0.02, num_leaves=64,
                  min_child_samples=80, feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1,
                  lambda_l1=1.0, lambda_l2=2.0, verbosity=-1, n_jobs=-1, scale_pos_weight=spw)

vm = (pd.to_datetime(train["decision_day"]) >= pd.Timestamp("2025-03-01")).values
p = dict(lgb_params); p["seed"] = SEED
dtr = lgb.Dataset(X1tr[~vm], y[~vm], categorical_feature=CAT)
dva = lgb.Dataset(X1tr[vm], y[vm], categorical_feature=CAT, reference=dtr)
m = lgb.train(p, dtr, 2500, valid_sets=[dva], callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
print("time-holdout AUC (LightGBM):", round(roc_auc_score(y[vm], m.predict(X1tr[vm])), 5))

time-holdout AUC (LightGBM): 0.75101


## 5. Вариант 1 — одиночная модель (бейзлайн)

In [8]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
idx = [feats.index(c) for c in CAT]
Xc = X1tr.copy(); Xc[CAT] = Xc[CAT].astype(object).fillna("NA")
oof_cb = np.zeros(len(train))
for tri, vai in skf.split(X1tr, y):
    cb = CatBoostClassifier(iterations=2000, learning_rate=0.03, depth=7, l2_leaf_reg=5,
                            loss_function="Logloss", eval_metric="AUC", random_seed=SEED,
                            scale_pos_weight=spw, verbose=0, early_stopping_rounds=120)
    cb.fit(Pool(Xc.iloc[tri], y.iloc[tri], cat_features=idx),
           eval_set=Pool(Xc.iloc[vai], y.iloc[vai], cat_features=idx))
    oof_cb[vai] = cb.predict_proba(Xc.iloc[vai])[:, 1]
print("одиночный CatBoost OOF AUC:", round(roc_auc_score(y, oof_cb), 5))

одиночный CatBoost OOF AUC: 0.83408


## 6. Вариант 2 — ансамбль 3 моделей (ранговый бленд)

LightGBM + CatBoost + XGBoost. Раньше усредняли вероятности и держали по нескольку
моделей каждого типа; перешли на ранговый бленд (у моделей разные шкалы, а для
AUC важен только порядок) и оставили по одной модели каждого типа.

In [9]:
def run_lgb(base, Xtr, Xte, skf):
    oof, tst = np.zeros(len(Xtr)), np.zeros(len(Xte))
    for f, (tri, vai) in enumerate(skf.split(Xtr, y)):
        p = dict(lgb_params); p["seed"] = base + 100 + f
        dt = lgb.Dataset(Xtr.iloc[tri], y.iloc[tri], categorical_feature=CAT)
        dv = lgb.Dataset(Xtr.iloc[vai], y.iloc[vai], categorical_feature=CAT, reference=dt)
        ml = lgb.train(p, dt, 2500, valid_sets=[dv], callbacks=[lgb.early_stopping(150, verbose=False), lgb.log_evaluation(0)])
        oof[vai] = ml.predict(Xtr.iloc[vai]); tst += ml.predict(Xte) / N_FOLDS
    return oof, tst

def run_cb(base, Xtr, Xte, skf):
    Xtr, Xte = Xtr.copy(), Xte.copy()
    Xtr[CAT] = Xtr[CAT].astype(object).fillna("NA"); Xte[CAT] = Xte[CAT].astype(object).fillna("NA")
    ci = [list(Xtr.columns).index(c) for c in CAT]
    oof, tst = np.zeros(len(Xtr)), np.zeros(len(Xte))
    for f, (tri, vai) in enumerate(skf.split(Xtr, y)):
        m = CatBoostClassifier(iterations=2000, learning_rate=0.03, depth=7, l2_leaf_reg=5,
                               loss_function="Logloss", eval_metric="AUC", random_seed=base + 100 + f,
                               scale_pos_weight=spw, verbose=0, early_stopping_rounds=120)
        m.fit(Pool(Xtr.iloc[tri], y.iloc[tri], cat_features=ci), eval_set=Pool(Xtr.iloc[vai], y.iloc[vai], cat_features=ci))
        oof[vai] = m.predict_proba(Xtr.iloc[vai])[:, 1]; tst += m.predict_proba(Xte)[:, 1] / N_FOLDS
    return oof, tst

def run_xgb(base, Xtr, Xte, skf):
    Xtr, Xte = Xtr.copy(), Xte.copy()
    for c in CAT:
        Xtr[c] = Xtr[c].cat.codes.replace(-1, np.nan); Xte[c] = Xte[c].cat.codes.replace(-1, np.nan)
    dte = xgb.DMatrix(Xte)
    oof, tst = np.zeros(len(Xtr)), np.zeros(len(Xte))
    for f, (tri, vai) in enumerate(skf.split(Xtr, y)):
        xp = dict(objective="binary:logistic", eval_metric="auc", eta=0.02, max_depth=6, subsample=0.8,
                  colsample_bytree=0.7, min_child_weight=5, reg_lambda=2.0, reg_alpha=1.0,
                  scale_pos_weight=spw, seed=base + 100 + f)
        bst = xgb.train(xp, xgb.DMatrix(Xtr.iloc[tri], label=y.iloc[tri]), 3000,
                        evals=[(xgb.DMatrix(Xtr.iloc[vai], label=y.iloc[vai]), "v")],
                        early_stopping_rounds=150, verbose_eval=False)
        oof[vai] = bst.predict(xgb.DMatrix(Xtr.iloc[vai])); tst += bst.predict(dte) / N_FOLDS
    return oof, tst

def ensemble(seed):
    sk = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof, tst = {}, {}
    oof["lgb"], tst["lgb"] = run_lgb(seed, X1tr, X1te, sk)
    oof["cat"], tst["cat"] = run_cb(seed, X1tr, X1te, sk)
    oof["xgb"], tst["xgb"] = run_xgb(seed, X1tr, X1te, sk)
    return oof, tst

oof2, tst2 = ensemble(SEED)
for k in oof2:
    print(f"OOF AUC {k}: {roc_auc_score(y, oof2[k]):.5f}")
print("OOF AUC бленд (rank-avg):", round(roc_auc_score(y, np.mean([rankdata(oof2[k]) for k in oof2], axis=0)), 5))

OOF AUC lgb: 0.83349
OOF AUC cat: 0.83408
OOF AUC xgb: 0.83324
OOF AUC бленд (rank-avg): 0.83649


## 7. Вариант 3 — + seed-bagging (финал)

Повторяем ансамбль с несколькими сидами разбиения и усредняем oof/test — снимает
случайный шум конкретного фолда.

In [10]:
oof_acc = {k: np.zeros(len(train)) for k in ["lgb", "cat", "xgb"]}
tst_acc = {k: np.zeros(len(test)) for k in ["lgb", "cat", "xgb"]}
for s in BAG_SEEDS:
    o, t = ensemble(s)
    for k in oof_acc:
        oof_acc[k] += o[k] / len(BAG_SEEDS)
        tst_acc[k] += t[k] / len(BAG_SEEDS)
    print(f"seed {s}: rank-avg OOF {roc_auc_score(y, np.mean([rankdata(o[k]) for k in o], axis=0)):.5f}")

oof_blend = np.mean([rankdata(oof_acc[k]) for k in oof_acc], axis=0)
print("seed-bagged OOF AUC:", round(roc_auc_score(y, oof_blend), 5))

seed 42: rank-avg OOF 0.83649
seed 202: rank-avg OOF 0.83631
seed 707: rank-avg OOF 0.83672
seed-bagged OOF AUC: 0.83741


## 8. Сабмит

Для test — усреднённые прогнозы, для id только из train — их out-of-fold значения.
Финальный скор — ранг по всему пулу, нормированный в (0, 1).

In [11]:
need_overlap = set(sample[ID].values) - set(test[ID].values)
parts = []
for k in oof_acc:
    te_part = pd.DataFrame({ID: test[ID].values, k: tst_acc[k]})
    ov_part = pd.DataFrame({ID: train[ID].values, k: oof_acc[k]})
    ov_part = ov_part[ov_part[ID].isin(need_overlap)]
    parts.append(pd.concat([te_part, ov_part], ignore_index=True))
allp = parts[0]
for pt in parts[1:]:
    allp = allp.merge(pt, on=ID)
allp["score"] = np.mean([rankdata(allp[k].values) for k in oof_acc], axis=0)

submission = sample[[ID]].merge(allp[[ID, "score"]], on=ID, how="left").rename(columns={"score": TARGET})
submission[TARGET] = submission[TARGET] / (len(submission) + 1)
assert submission[TARGET].isna().sum() == 0 and len(submission) == len(sample)
submission.to_csv("submission.csv", index=False)
print("saved submission.csv:", submission.shape)
submission.head()

saved submission.csv: (45032, 2)


   front_id  target_value
0    238459      0.070637
1    151533      0.454977
2    195027      0.665727
3    195034      0.741871
4    129620      0.668073

## Что ещё пробовал

- **target encoding и произведения признаков** — поднимали OOF, но итоговый лучший вариант лучше работает без него
  (переобучение под прошлый период); оставлены опцией `USE_TE`;
- **простое среднее вероятностей** вместо рангового — чуть хуже из-за разных шкал;
- **больше моделей одного типа** (было 7) — почти не добавляли разнообразия, хотя изначально дали неплохой результат;
- **хронологическая валидация для отбора** — занижала и плохо коррелировала с тестом;
- **5 сидов вместо 3** — отдача уже в пределах шума.